# Ablation Study : RAG baseline vs LightRAG hybride vs Agentic GraphRAG

**PFE Agentic GraphRAG — Wiame Anejjar — Master SDIA 2025-2026**

## Objectif

Isoler la contribution de chaque couche de l'architecture en évaluant **trois systèmes** sur le **même benchmark**, avec le **même LLM générateur** et les **mêmes métriques RAGAS** :

1. **RAG vectoriel (ChromaDB seul)** — aucun graphe, retrieval par similarité vectorielle uniquement.
2. **LightRAG hybride (sans agent)** — retrieval graphe + vecteurs de LightRAG (mode hybrid), une seule génération, sans boucle de correction.
3. **Agentic GraphRAG (contribution du PFE)** — même retrieval LightRAG que (2), plus la boucle CRITIQUE → SELF_CORRECT avec juge indépendant.

La seule variable qui change entre (2) et (3) est la présence de la couche agentique : cela permet de mesurer sa contribution propre, isolée du choix du backend de retrieval.

**Benchmark utilisé** : `data/processed/benchmark_true_multihop.json` — 76 questions vrai multi-hop, validées par analyse de contenu (voir `scripts/validate_multihop_benchmark.py`), pas par la simple étiquette `hop_type` d'origine.

**Important** : les résultats de ce notebook sont ceux effectivement obtenus à l'exécution , aucun chiffre n'est pré-rempli ou supposé à l'avance.

In [1]:
import os, sys, json, random, asyncio, time, datetime
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import nest_asyncio
nest_asyncio.apply()

from dotenv import load_dotenv
load_dotenv()

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline
pd.set_option("display.max_colwidth", 120)
print(f"Notebook exécuté le : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Notebook exécuté le : 2026-07-24 00:33:29


## 1. Chargement du benchmark

In [2]:
BENCHMARK_PATH = Path("data/processed/benchmark_true_multihop.json")
# Recommandation prof : réduire l'échelle le temps de stabiliser le scoring
# RAGAS (0 erreur / 0 NaN sur ce volume), puis remonter à 20 progressivement.
N_QUESTIONS = 10
SEED = 42

with BENCHMARK_PATH.open("r", encoding="utf-8") as f:
    all_items = json.load(f)

random.seed(SEED)
eval_items = random.sample(all_items, min(N_QUESTIONS, len(all_items)))

print(f"Benchmark source                     : {BENCHMARK_PATH}")
print(f"Questions vrai multi-hop disponibles  : {len(all_items)}")
print(f"Questions échantillonnées (seed={SEED})    : {len(eval_items)}")

display(pd.DataFrame([
    {"question": it["question"], "ground_truth": it["ground_truth"]}
    for it in eval_items
]))

Benchmark source                     : data\processed\benchmark_true_multihop.json
Questions vrai multi-hop disponibles  : 76
Questions échantillonnées (seed=42)    : 10


,question,ground_truth
0,"What type of knowledge graph model is used to represent context-dependent triplet validity, which is developed from ...",Quantum Knowledge Graph
1,What method is designed to improve defenses against the type of models that often rely on the Model Context Protocol...,Autorise Method
2,"What system is built on Gemini 2.5 Flash, which exhibits a lower but still significant individualism-collectivism bi...",Itas
3,What type of cultural bias is exhibited by the LLM applied to synthetic patient profiles in reliability auditing?,Western-Style Individualism
4,"What aspect of GPT-5.4 is affected when generic retry is applied to improve its performance, and this model only res...",GPT-5.4's Stated Country Identity
5,What approach is based on the core idea of a learning method that Hindsight Preference Optimization draws from to ge...,Epm-Rl
6,What type of models have shown improved image editing performance by incorporating the method used to solve math pro...,Unified Multi-Modal Understanding/Generative Models
7,What is the application of the method that outperforms UniVL-DR on the WebQA+ and EVQA+ datasets when considering sh...,Target Binding
8,What phenomenon occurs in TTRL as a result of the challenges posed by label noise in medical image classification ad...,Spurious Signal Amplification
9,"What aspect of performance do Vision-Language Models, which utilize Bayesian Inference Module for anomaly localizati...",Temporal Consistency


## 2. Configuration commune (générateur + prompt)

Pour que la comparaison isole bien la variable étudiée (retrieval + boucle agentique), les trois systèmes réutilisent **directement** le générateur et le prompt de `src/agent/graph_v3.py` (fonction `_llm_call`, prompt `RESPONSE_SYSTEM_PROMPT`) — pas une copie séparée. Quel que soit le backend actif (NVIDIA / Groq / Ollama local, contrôlé par les variables d'environnement `USE_NVIDIA`/`USE_GROQ`), les trois systèmes utilisent donc exactement le même LLM pour la génération finale.

Seuls le contexte fourni (retrieval) et la présence ou non de la boucle CRITIQUE/SELF_CORRECT diffèrent entre les trois systèmes.

In [3]:
from langchain_ollama import OllamaEmbeddings

# Réutilise l'infrastructure déjà initialisée dans graph_v3.py (déclenche l'init
# du LLM générateur, du juge et de rag_instance -> un seul point de configuration)
from src.agent import graph_v3

EMBED_MODEL    = os.getenv("EMBED_MODEL", "nomic-embed-text")
OLLAMA_URL     = os.getenv("OLLAMA_URL", "http://localhost:11434")

if graph_v3.USE_NVIDIA and graph_v3.NVIDIA_API_KEY:
    GENERATOR_DESC = f"NVIDIA {graph_v3.NVIDIA_MODEL}"
elif graph_v3.USE_GROQ and graph_v3.GROQ_API_KEY:
    GENERATOR_DESC = f"Groq {graph_v3.GROQ_GENERATOR_MODEL}"
else:
    GENERATOR_DESC = f"Ollama {graph_v3.MODEL_NAME} (num_ctx={graph_v3.OLLAMA_NUM_CTX})"

def generate_answer(question: str, context: str) -> str:
    # Meme troncature defensive que node_response (evite les 413 "request too
    # large" de certains backends, ex. Groq llama-3.1-8b-instant)
    context = context[:graph_v3.MAX_GENERATOR_CONTEXT_CHARS]
    user = f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer (use ONLY the context above, cite sources):"
    return graph_v3._llm_call(graph_v3.RESPONSE_SYSTEM_PROMPT, user)

print(f"Générateur commun (systèmes 1, 2 et 3) : {GENERATOR_DESC}")
print(f"Contexte tronqué à {graph_v3.MAX_GENERATOR_CONTEXT_CHARS} caractères avant génération (anti-413)")

c:\Users\ADMIN\Desktop\PFE_Agentic_Graphrag\GraphRag_PFE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[LLM GENERATOR] Groq llama-3.1-8b-instant
[LLM JUDGE] Groq llama-3.3-70b-versatile (indépendant du générateur)
======= JUDGE STATUS ========== independent=True


INFO: [] Loaded graph from indexes\lightrag_500_connected_v2\graph_chunk_entity_relation.graphml with 4945 nodes, 5593 edges
INFO:nano-vectordb:Load (4980, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': 'indexes\\lightrag_500_connected_v2\\vdb_entities.json'} 4980 data
INFO:nano-vectordb:Load (5730, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': 'indexes\\lightrag_500_connected_v2\\vdb_relationships.json'} 5730 data
INFO:nano-vectordb:Load (500, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': 'indexes\\lightrag_500_connected_v2\\vdb_chunks.json'} 500 data
INFO: [] Process 35428 KV load full_docs with 500 records
INFO: [] Process 35428 KV load text_chunks with 500 records
INFO: [] Process 35428 KV load full_entities with 500 records
INFO: [] Process 35428 KV load full_relations with 5068 records
INFO: [] Process 35428 KV load entity_chunks with 4977 re

[LIGHTRAG] Initialisé → indexes\lightrag_500_connected_v2
Générateur commun (systèmes 1, 2 et 3) : Groq llama-3.1-8b-instant
Contexte tronqué à 6000 caractères avant génération (anti-413)


## 3. Système 1 — RAG vectoriel (ChromaDB seul)

In [4]:
from langchain_chroma import Chroma

CHROMA_DIR        = os.getenv("CHROMA_DIR", "indexes/chroma_pfe500_baseline")
CHROMA_COLLECTION = "pfe_500_baseline"
TOP_K_VECTOR      = int(os.getenv("TOP_K_VECTOR", "5"))

_baseline_emb = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_URL)
_vectorstore  = Chroma(persist_directory=CHROMA_DIR, embedding_function=_baseline_emb,
                       collection_name=CHROMA_COLLECTION)
_baseline_retriever = _vectorstore.as_retriever(search_kwargs={"k": TOP_K_VECTOR})

print(f"ChromaDB         : {CHROMA_DIR} | collection={CHROMA_COLLECTION}")
print(f"Vecteurs stockés : {_vectorstore._collection.count()}")
print(f"top_k            : {TOP_K_VECTOR}")

def run_rag_baseline(question: str) -> dict:
    t0 = time.time()
    docs = _baseline_retriever.invoke(question)
    contexts = [d.page_content for d in docs]
    context_str = "\n\n".join(f"[Doc {i+1}] {c[:1200]}" for i, c in enumerate(contexts))
    answer = generate_answer(question, context_str)
    return {"answer": answer,
            "contexts": contexts if contexts else ["No context retrieved."],
            "context_chars": len(context_str),
            "latency_s": round(time.time() - t0, 2)}

ChromaDB         : indexes/chroma_pfe500_baseline | collection=pfe_500_baseline
Vecteurs stockés : 1587
top_k            : 5


In [5]:
results_baseline = []
ragas_data_baseline = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

print("=== SYSTÈME 1 : RAG BASELINE (ChromaDB) ===")
for i, ex in enumerate(eval_items, 1):
    print(f"[{i}/{len(eval_items)}] {ex['question'][:70]}")
    r = run_rag_baseline(ex["question"])
    ragas_data_baseline["question"].append(ex["question"])
    ragas_data_baseline["answer"].append(r["answer"])
    ragas_data_baseline["contexts"].append(r["contexts"])
    ragas_data_baseline["ground_truth"].append(ex["ground_truth"])
    results_baseline.append({"question": ex["question"], "answer": r["answer"][:200],
                              "context_chars": r["context_chars"], "latency_s": r["latency_s"]})
    print(f"  -> {r['answer'][:100]}  ({r['latency_s']}s)")

df_details_baseline = pd.DataFrame(results_baseline)

=== SYSTÈME 1 : RAG BASELINE (ChromaDB) ===
[1/10] What type of knowledge graph model is used to represent context-depend
  -> The type of knowledge graph model used to represent context-dependent triplet validity is referred t  (42.1s)
[2/10] What method is designed to improve defenses against the type of models
  -> The corpus does not contain information to answer this question.  (8.54s)
[3/10] What system is built on Gemini 2.5 Flash, which exhibits a lower but s
  -> The corpus does not contain information to answer this question.  (0.35s)
[4/10] What type of cultural bias is exhibited by the LLM applied to syntheti
  -> The corpus does not contain information to answer this question.  (0.36s)
[5/10] What aspect of GPT-5.4 is affected when generic retry is applied to im
  -> The corpus does not contain information to answer this question.  (0.38s)
[6/10] What approach is based on the core idea of a learning method that Hind


INFO:groq._base_client:Retrying request to /openai/v1/chat/completions in 6.000000 seconds


  -> The corpus does not contain information to answer this question.  (6.62s)
[7/10] What type of models have shown improved image editing performance by i
  -> The models that have shown improved image editing performance by incorporating the method used to so  (2.54s)
[8/10] What is the application of the method that outperforms UniVL-DR on the


INFO:groq._base_client:Retrying request to /openai/v1/chat/completions in 10.000000 seconds


  -> The corpus does not contain information to answer this question.  (10.57s)
[9/10] What phenomenon occurs in TTRL as a result of the challenges posed by 


INFO:groq._base_client:Retrying request to /openai/v1/chat/completions in 3.000000 seconds


  -> The phenomenon that occurs in TTRL (Transferable Robust Learning) as a result of the challenges pose  (7.95s)
[10/10] What aspect of performance do Vision-Language Models, which utilize Ba


INFO:groq._base_client:Retrying request to /openai/v1/chat/completions in 8.000000 seconds


  -> The corpus does not contain information to answer this question.  (8.57s)


## 4. Système 2 — LightRAG hybride (retrieval seul, sans agent)

Réutilise directement `rag_instance` de `src/agent/graph_v3.py` (même index post-traité, mêmes paramètres `top_k`/`chunk_top_k`/mode) — garantit un retrieval strictement identique à celui du système 3. Seule différence : un unique appel de génération, sans CRITIQUE ni SELF_CORRECT.

In [6]:
from lightrag import QueryParam

print(f"[LightRAG] index réutilisé : {graph_v3.INDEX_DIR}")
print(f"[LightRAG] top_k={graph_v3.TOP_K_LIGHTRAG} | chunk_top_k={graph_v3.CHUNK_TOP_K} | mode=hybrid")

def run_lightrag_only(question: str) -> dict:
    t0 = time.time()

    async def _query():
        return await graph_v3.rag_instance.aquery_data(
            question,
            param=QueryParam(
                mode="hybrid",
                top_k=graph_v3.TOP_K_LIGHTRAG,
                chunk_top_k=graph_v3.CHUNK_TOP_K,
                enable_rerank=False,
                max_total_tokens=int(os.getenv("MAX_TOTAL_TOKENS", "12000")),
            ),
        )

    result = asyncio.get_event_loop().run_until_complete(_query())
    data = (result or {}).get("data", {}) if isinstance(result, dict) else {}
    context_str = graph_v3._format_context_from_raw(data)
    contexts    = graph_v3._contexts_list_from_raw(data)
    answer = generate_answer(question, context_str)  # UN SEUL appel, pas de critique/self-correct
    return {"answer": answer, "contexts": contexts,
            "context_chars": len(context_str),
            "latency_s": round(time.time() - t0, 2)}

[LightRAG] index réutilisé : indexes\lightrag_500_connected_v2
[LightRAG] top_k=15 | chunk_top_k=10 | mode=hybrid


In [7]:
results_lightrag = []
ragas_data_lightrag = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

print("=== SYSTÈME 2 : LIGHTRAG HYBRIDE (SANS AGENT) ===")
for i, ex in enumerate(eval_items, 1):
    print(f"[{i}/{len(eval_items)}] {ex['question'][:70]}")
    r = run_lightrag_only(ex["question"])
    ragas_data_lightrag["question"].append(ex["question"])
    ragas_data_lightrag["answer"].append(r["answer"])
    ragas_data_lightrag["contexts"].append(r["contexts"])
    ragas_data_lightrag["ground_truth"].append(ex["ground_truth"])
    results_lightrag.append({"question": ex["question"], "answer": r["answer"][:200],
                              "context_chars": r["context_chars"], "latency_s": r["latency_s"]})
    print(f"  -> {r['answer'][:100]}  ({r['latency_s']}s)")

df_details_lightrag = pd.DataFrame(results_lightrag)
print(f"\nTaille moyenne du contexte fusionné : {df_details_lightrag['context_chars'].mean():.0f} caractères "
      f"(max={df_details_lightrag['context_chars'].max()})")

INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)


=== SYSTÈME 2 : LIGHTRAG HYBRIDE (SANS AGENT) ===
[1/10] What type of knowledge graph model is used to represent context-depend


INFO: Query nodes: MindTrellis, AI-assisted knowledge graph creation, Triplet validation (top_k:15, cosine:0.2)
INFO: Local query: 15 entites, 56 relations
INFO: Query edges: Knowledge graph model, Context-dependent triplet validity, Collaborative construction (top_k:15, cosine:0.2)
INFO: Global query: 21 entites, 14 relations
INFO: Raw search results: 30 entities, 60 relations, 0 vector chunks
INFO: After truncation: 30 entities, 60 relations
INFO: Selecting 75 from 136 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 60 relations
INFO: Round-robin merged chunks: 75 -> 75 (deduplicated 0)
INFO: Final context: 30 entities, 60 relations, 10 chunks
INFO: Final chunks S+F/O: E5/1 E3/2 E3/3 E5/4 E1/5 E3/6 E1/7 E2/8 E1/9 E1/10


  -> The Quantum Knowledge Graph model is used to represent context-dependent triplet validity [model: Qu  (3.08s)
[2/10] What method is designed to improve defenses against the type of models


INFO: Query nodes: Model Context Protocol, Defensive methods (top_k:15, cosine:0.2)
INFO: Local query: 14 entites, 32 relations
INFO: Query edges: Model defense, External tool connections (top_k:15, cosine:0.2)
INFO: Global query: 24 entites, 14 relations
INFO: Raw search results: 37 entities, 45 relations, 0 vector chunks
INFO: After truncation: 37 entities, 45 relations
INFO: Selecting 21 from 21 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 45 relations
INFO: Round-robin merged chunks: 21 -> 21 (deduplicated 0)
INFO: Final context: 37 entities, 45 relations, 10 chunks
INFO: Final chunks S+F/O: E2/1 E2/2 E1/3 E1/4 E2/5 E2/6 E2/7 E2/8 E4/9 E2/10


...

  -> The corpus does not contain information to answer this question.  (23.53s)
[9/10] What phenomenon occurs in TTRL as a result of the challenges posed by 


INFO: Query nodes: TTRL, Phenomenon, Challenges, Robust learning (top_k:15, cosine:0.2)
INFO: Local query: 15 entites, 64 relations
INFO: Query edges: Label noise, Medical image classification, Risk-aware robust learning (top_k:15, cosine:0.2)
INFO: Global query: 16 entites, 14 relations
INFO: Raw search results: 31 entities, 78 relations, 0 vector chunks
INFO: After truncation: 31 entities, 78 relations
INFO: Selecting 32 from 32 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 78 relations
INFO: Round-robin merged chunks: 32 -> 32 (deduplicated 0)
INFO: Final context: 31 entities, 78 relations, 10 chunks
INFO: Final chunks S+F/O: E9/1 E2/2 E2/3 E1/4 E5/5 E1/6 E1/7 E1/8 E1/9 E1/10
INFO:groq._base_client:Retrying request to /openai/v1/chat/completions in 20.000000 seconds


  -> The phenomenon that occurs in TTRL as a result of the challenges posed by label noise in medical ima  (23.18s)
[10/10] What aspect of performance do Vision-Language Models, which utilize Ba


INFO: Query nodes: Performance metrics, Dynamic environments, Real-world scenarios, Anomaly detection (top_k:15, cosine:0.2)
INFO: Local query: 15 entites, 29 relations
INFO: Query edges: Vision-Language Models, Bayesian Inference Module, Anomaly localization (top_k:15, cosine:0.2)
INFO: Global query: 20 entites, 15 relations
INFO: Raw search results: 34 entities, 42 relations, 0 vector chunks
INFO: After truncation: 34 entities, 42 relations
INFO: Selecting 85 from 165 entity-related chunks by vector similarity
INFO: Find 5 additional chunks in 5 relations (deduplicated 18)
INFO: Selecting 5 from 5 relation-related chunks by vector similarity
INFO: Round-robin merged chunks: 90 -> 90 (deduplicated 0)
INFO: Final context: 34 entities, 42 relations, 10 chunks
INFO: Final chunks S+F/O: E2/1 R1/1 E2/2 R2/2 E1/3 R1/3 E1/4 R1/4 E1/5 R1/5
INFO:groq._base_client:Retrying request to /openai/v1/chat/completions in 4.000000 seconds


  -> Vision-Language Models, which utilize Bayesian Inference Module for anomaly localization, often stru  (5.73s)

Taille moyenne du contexte fusionné : 30062 caractères (max=39719)


## 5. Système 3 — Agentic GraphRAG (contribution du PFE)

Boucle complète : QUERY → HYBRID_SEARCH (LightRAG, identique au système 2) → RESPONSE → CRITIQUE (juge indépendant) → FINALIZE | SELF_CORRECT (max 3 itérations). Voir `src/agent/graph_v3.py::run_agent`.

In [8]:
results_agentic = []
ragas_data_agentic = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

print("=== SYSTÈME 3 : AGENTIC GRAPHRAG ===")
for i, ex in enumerate(eval_items, 1):
    print(f"[{i}/{len(eval_items)}] {ex['question'][:70]}")
    t0 = time.time()
    result = graph_v3.run_agent(ex["question"])
    latency = round(time.time() - t0, 2)

    ragas_data_agentic["question"].append(ex["question"])
    ragas_data_agentic["answer"].append(result.get("final_response", ""))
    ragas_data_agentic["contexts"].append(result.get("lightrag_retrieved_contexts") or ["No context retrieved."])
    ragas_data_agentic["ground_truth"].append(ex["ground_truth"])

    results_agentic.append({
        "question": ex["question"],
        "answer": result.get("final_response", "")[:200],
        "context_chars": len(result.get("lightrag_context", "")),
        "critique_score": result.get("critique_score", 0.0),
        "judge_independent": result.get("critique_judge_independent", False),
        "iterations": result.get("iteration", 0),
        "latency_s": latency,
    })
    print(f"  -> score={result.get('critique_score', 0):.2f} | iter={result.get('iteration', 0)} ({latency}s)")

df_details_agentic = pd.DataFrame(results_agentic)
n_self_eval = (~df_details_agentic["judge_independent"]).sum()
if n_self_eval:
    print(f"\n⚠ {n_self_eval}/{len(df_details_agentic)} scores de critique sont des auto-évaluations (judge_independent=False)")

=== SYSTÈME 3 : AGENTIC GRAPHRAG ===
[1/10] What type of knowledge graph model is used to represent context-depend

QUESTION : What type of knowledge graph model is used to represent context-dependent triplet validity, which is developed from the knowledge graph created through collaborative construction with AI in MindTrellis?

[QUERY] iter=0 | What type of knowledge graph model is used to represent context-dependent triplet validity, which is developed from the knowledge graph created through collaborative construction with AI in MindTrellis?


INFO: Query nodes: MindTrellis, AI-assisted knowledge graph creation, Triplet validation (top_k:15, cosine:0.2)
INFO: Local query: 15 entites, 56 relations
INFO: Query edges: Knowledge graph model, Context-dependent triplet validity, Collaborative construction (top_k:15, cosine:0.2)
INFO: Global query: 21 entites, 14 relations
INFO: Raw search results: 30 entities, 60 relations, 0 vector chunks
INFO: After truncation: 30 entities, 60 relations
INFO: Selecting 75 from 136 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 60 relations
INFO: Round-robin merged chunks: 75 -> 75 (deduplicated 0)
INFO: Final context: 30 entities, 60 relations, 10 chunks
INFO: Final chunks S+F/O: E5/1 E3/2 E3/3 E5/4 E1/5 E3/6 E1/7 E2/8 E1/9 E1/10


[HYBRID_SEARCH] context=33017 chars | 100 passages RAGAS
[RESPONSE iter=0] The Quantum Knowledge Graph model is used to represent context-dependent triplet validity [model: Quantum Knowledge Graph]. 

This model formulates tr
[CRITIQUE iter=0] score=0.00 | judge_independent=True
[ROUTE] score=0.00 < 0.75, iter=0/3 → SELF_CORRECT
[SELF_CORRECT] 0→1 | What type of knowledge graph model is specifically used in MindTrellis to represent context-dependent triplet validity, and what is the nature of the knowledge graph that it is based on, which was created through collaborative construction with AI?

[QUERY] iter=1 | What type of knowledge graph model is specifically used in MindTrellis to represent context-dependent triplet validity, and what is the nature of the knowledge graph that it is based on, which was created through collaborative construction with AI?


INFO: Query nodes: MindTrellis, Context-dependent, Triplet validity, Knowledge graph, Collaborative construction, AI (top_k:15, cosine:0.2)
INFO: Local query: 15 entites, 62 relations
INFO: Query edges: Knowledge graph model, Context-dependent triplet validity, Collaborative construction with AI (top_k:15, cosine:0.2)
INFO: Global query: 21 entites, 14 relations
INFO: Raw search results: 31 entities, 69 relations, 0 vector chunks
INFO: After truncation: 31 entities, 69 relations
INFO: Selecting 77 from 135 entity-related chunks by vector similarity
INFO: Find 1 additional chunks in 1 relations (deduplicated 15)
INFO: Selecting 1 from 1 relation-related chunks by vector similarity
INFO: Round-robin merged chunks: 78 -> 78 (deduplicated 0)
INFO: Final context: 31 entities, 69 relations, 10 chunks
INFO: Final chunks S+F/O: E5/1 R1/1 E3/2 E3/3 E5/4 E1/5 E1/6 E3/7 E1/8 E2/9


[HYBRID_SEARCH] context=33999 chars | 110 passages RAGAS
[RESPONSE iter=1] The knowledge graph model specifically used in MindTrellis to represent context-dependent triplet validity is not explicitly mentioned in the provided
[CRITIQUE iter=1] score=0.85 | judge_independent=True
[ROUTE] score=0.85 >= 0.75 → FINALIZE
[FINALIZE] score=0.85 après 1 iteration(s)

--- RÉPONSE FINALE ---
The knowledge graph model specifically used in MindTrellis to represent context-dependent triplet validity is not explicitly mentioned in the provided context. However, it is mentioned that MindTrellis is an interactive visual system that enables users and AI to collaboratively build a dynamic knowledge graph for organizing information from multiple documents [Doc N].

The nature of the knowledge graph that it is based on, which was created through collaborative construction with AI, is a dynamic representation of relationships between concepts used for organizing information [dataset: Knowledge Graph].
Scor

...

[SELF_CORRECT] 2→3 | What specific performance limitations or challenges do Vision-Language Models that incorporate a Bayesian Inference Module for anomaly localization typically face in dynamic real-world environments, as directly supported by relevant research or studies?

[QUERY] iter=3 | What specific performance limitations or challenges do Vision-Language Models that incorporate a Bayesian Inference Module for anomaly localization typically face in dynamic real-world environments, as directly supported by relevant research or studies?


INFO:groq._base_client:Retrying request to /openai/v1/chat/completions in 8.000000 seconds
INFO:  == LLM cache == saving: hybrid:keywords:997cce837ccfe94ea8d167a021f9db5e
INFO: Query nodes: Performance limitations, Challenges, Research studies, Anomaly detection, Machine learning, Computer vision, Natural language processing (top_k:15, cosine:0.2)
INFO: Local query: 15 entites, 73 relations
INFO: Query edges: Vision-Language Models, Bayesian Inference Module, Anomaly localization, Dynamic real-world environments (top_k:15, cosine:0.2)
INFO: Global query: 20 entites, 15 relations
INFO: Raw search results: 34 entities, 82 relations, 0 vector chunks
INFO: After truncation: 34 entities, 82 relations
INFO: Selecting 85 from 167 entity-related chunks by vector similarity
INFO: Find 9 additional chunks in 9 relations (deduplicated 25)
INFO: Selecting 9 from 9 relation-related chunks by vector similarity
INFO: Round-robin merged chunks: 94 -> 94 (deduplicated 0)
INFO: Final context: 34 entitie

[HYBRID_SEARCH] context=42892 chars | 125 passages RAGAS


INFO:groq._base_client:Retrying request to /openai/v1/chat/completions in 13.000000 seconds


[RESPONSE iter=3] Vision-Language Models (VLMs) that incorporate a Bayesian Inference Module for anomaly localization typically face the challenge of object hallucinati
[CRITIQUE iter=3] score=0.40 | judge_independent=True
[ROUTE] MAX_ITERATIONS atteint → FINALIZE
[FINALIZE] score=0.40 après 3 iteration(s)

--- RÉPONSE FINALE ---
Vision-Language Models (VLMs) that incorporate a Bayesian Inference Module for anomaly localization typically face the challenge of object hallucination due to an over-reliance on linguistic priors, which can lead to inaccurate results in dynamic real-world environments [Graph: Vision-Language Models]. 

Additionally, VLMs struggle with interactive causal learning and transferring latent structures across contexts, which can further limit their performance in dynamic real-world scenarios [Graph: Vision-Language Models].
Score : 0.40 | judge_independent=True | Iterations : 3
  -> score=0.40 | iter=3 (75.16s)


## 6. Évaluation RAGAS des 3 systèmes

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.run_config import RunConfig
from langchain_ollama import ChatOllama

# Recommandations prof suite aux NaN observés avec batch_size=4 sur Groq :
# le problème vient de la CONCURRENCE + de l'instabilité du juge (429 / JSON
# mal formé), pas du pipeline lui-même. Corrections :
#  1) concurrence RAGAS abaissée (max_workers=2, batch_size réduit)
#  2) back-off plus généreux (max_retries, max_wait augmentés)
#  3) juge API plus fiable en option si Groq reste instable -> Google Gemini
#     (gratuit, USE_GEMINI_JUDGE + GEMINI_API_KEY) en priorité
#  4) échelle réduite (N_QUESTIONS=10, cf. section 1) le temps de stabiliser
#  5) diagnostic NaN après coup 
_use_gemini_judge = os.getenv("USE_GEMINI_JUDGE", "false").lower() == "true"
_gemini_key       = os.getenv("GEMINI_API_KEY", "")
_use_openai_judge = os.getenv("USE_OPENAI_JUDGE", "false").lower() == "true"
_openai_key       = os.getenv("OPENAI_API_KEY", "")

if _use_gemini_judge and _gemini_key:
    from langchain_google_genai import ChatGoogleGenerativeAI
    _gemini_judge_model = os.getenv("GEMINI_JUDGE_MODEL", "gemini-2.0-flash")
    print(f"[RAGAS JUDGE] Google Gemini {_gemini_judge_model} (juge API fiable, gratuit, fallback Groq)")
    _ragas_chat_model = ChatGoogleGenerativeAI(model=_gemini_judge_model, google_api_key=_gemini_key, temperature=0)
    _ragas_batch_size = 2
elif _use_openai_judge and _openai_key:
    from langchain_openai import ChatOpenAI
    _openai_judge_model = os.getenv("OPENAI_JUDGE_MODEL", "gpt-4o-mini")
    print(f"[RAGAS JUDGE] OpenAI {_openai_judge_model} (juge API fiable, fallback Groq)")
    _ragas_chat_model = ChatOpenAI(model=_openai_judge_model, api_key=_openai_key, temperature=0)
    _ragas_batch_size = 2
elif graph_v3.USE_GROQ and graph_v3.GROQ_API_KEY:
    from langchain_groq import ChatGroq
    _ragas_groq_model = os.getenv("RAGAS_GROQ_MODEL", "llama-3.1-8b-instant")
    print(f"[RAGAS JUDGE] Groq {_ragas_groq_model}")
    _ragas_chat_model = ChatGroq(model=_ragas_groq_model, api_key=graph_v3.GROQ_API_KEY, temperature=0)
    _ragas_batch_size = 2  # abaissé de 4 à 2 (recommandation prof, anti-429)
elif graph_v3.USE_NVIDIA and graph_v3.NVIDIA_API_KEY:
    from langchain_openai import ChatOpenAI
    print(f"[RAGAS JUDGE] NVIDIA {graph_v3.NVIDIA_MODEL}")
    _ragas_chat_model = ChatOpenAI(model=graph_v3.NVIDIA_MODEL, api_key=graph_v3.NVIDIA_API_KEY,
                                    base_url=graph_v3.NVIDIA_BASE_URL, temperature=0)
    _ragas_batch_size = 1
else:
    print(f"[RAGAS JUDGE] Ollama {graph_v3.MODEL_NAME} (local)")
    _ragas_chat_model = ChatOllama(model=graph_v3.MODEL_NAME, base_url=OLLAMA_URL, temperature=0)
    _ragas_batch_size = 1

_ragas_llm = LangchainLLMWrapper(_ragas_chat_model)
_ragas_emb = LangchainEmbeddingsWrapper(OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_URL))
print(f"[RAGAS CONFIG] batch_size={_ragas_batch_size}, max_workers=2, max_retries=5, max_wait=60")

NAN_THRESHOLD_PCT = 25

def run_ragas(ragas_data: dict, label: str) -> pd.DataFrame:
    print(f"\nRAGAS -> {label} ({len(ragas_data['question'])} questions, batch_size={_ragas_batch_size})...")
    dataset = Dataset.from_dict(ragas_data)
    result = evaluate(
        dataset,
        metrics=[
            Faithfulness(llm=_ragas_llm),
            AnswerRelevancy(llm=_ragas_llm, embeddings=_ragas_emb),
            ContextPrecision(llm=_ragas_llm),
            ContextRecall(llm=_ragas_llm),
        ],
        run_config=RunConfig(timeout=180, max_retries=5, max_wait=60, max_workers=2),
        batch_size=_ragas_batch_size,
        raise_exceptions=False,
    )
    df = result.to_pandas()
    df["system"] = label

    # Recommandation prof (5) : compter les NaN par métrique avant toute
    # interprétation -> au-delà de 25% de NaN, la moyenne n'est pas exploitable.
    print(f"  Validité ({label}) :")
    for m in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
        n_nan = df[m].isna().sum()
        pct_nan = n_nan / len(df) * 100
        flag = "⚠ INEXPLOITABLE" if pct_nan > NAN_THRESHOLD_PCT else "OK"
        print(f"    {m:20s} : {n_nan}/{len(df)} NaN ({pct_nan:.0f}%) — {flag}")
    return df

df_ragas_baseline = run_ragas(ragas_data_baseline, "RAG baseline (ChromaDB)")
df_ragas_lightrag = run_ragas(ragas_data_lightrag, "LightRAG hybride (sans agent)")
df_ragas_agentic  = run_ragas(ragas_data_agentic,  "Agentic GraphRAG")

[RAGAS JUDGE] Google Gemini gemini-2.0-flash (juge API fiable, gratuit, fallback Groq)


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_35428\263425317.py:52: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  _ragas_llm = LangchainLLMWrapper(_ragas_chat_model)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_35428\263425317.py:53: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  _ragas_emb = LangchainEmbeddingsWrapper(OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_URL))


[RAGAS CONFIG] batch_size=2, max_workers=2, max_retries=5, max_wait=60

RAGAS -> RAG baseline (ChromaDB) (10 questions, batch_size=2)...


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:google_genai._api_client:Retrying google.genai._api_client.BaseApiClient._async_request_once in 1.67 seconds as it raised ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_reque

## 7. Tableau comparatif des résultats

In [ ]:
METRICS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
SYSTEM_ORDER = ["RAG baseline (ChromaDB)", "LightRAG hybride (sans agent)", "Agentic GraphRAG"]

all_ragas = pd.concat([df_ragas_baseline, df_ragas_lightrag, df_ragas_agentic], ignore_index=True)

comparison_table = all_ragas.groupby("system")[METRICS].mean().reindex(SYSTEM_ORDER).round(3)
comparison_table["latency_moy_s"] = [
    df_details_baseline["latency_s"].mean(),
    df_details_lightrag["latency_s"].mean(),
    df_details_agentic["latency_s"].mean(),
]

print(f"=== Tableau comparatif — moyennes RAGAS sur {len(eval_items)} questions ===")
display(comparison_table)

Path("Eval_agentic").mkdir(parents=True, exist_ok=True)
comparison_table.to_csv("Eval_agentic/ablation_study_comparaison.csv")
all_ragas.to_csv("Eval_agentic/ablation_study_details_par_question.csv", index=False)
print("\n✓ Eval_agentic/ablation_study_comparaison.csv")
print("✓ Eval_agentic/ablation_study_details_par_question.csv")

In [ ]:
ax = comparison_table[METRICS].plot(kind="bar", figsize=(11, 6), rot=15)
ax.set_ylabel("Score RAGAS (0-1)")
ax.set_title("Ablation study — RAG baseline vs LightRAG hybride vs Agentic GraphRAG")
ax.set_ylim(0, 1)
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()

Path("figures").mkdir(parents=True, exist_ok=True)
plt.savefig("figures/fig15_ablation_study_comparaison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ figures/fig15_ablation_study_comparaison.png")

## 8. Paramètres utilisés par système

In [ ]:
judge_desc = (
    f"Groq {graph_v3.GROQ_JUDGE_MODEL} (indépendant)"
    if graph_v3.USE_GROQ_JUDGE and graph_v3.GROQ_API_KEY
    else f"Ollama {graph_v3.JUDGE_MODEL_NAME} (indépendant)"
)

params_table = pd.DataFrame([
    {
        "Système": "RAG baseline (ChromaDB)",
        "Backend retrieval": f"ChromaDB — {CHROMA_DIR}",
        "Corpus indexé": f"{_vectorstore._collection.count()} chunks (nomic-embed-text, chunk_size=900, overlap=120, 500 documents)",
        "Paramètres retrieval": f"top_k={TOP_K_VECTOR}",
        "Générateur": GENERATOR_DESC,
        "Boucle agentique": "Non",
    },
    {
        "Système": "LightRAG hybride (sans agent)",
        "Backend retrieval": f"LightRAG — {graph_v3.INDEX_DIR} (post-traité : fusion entités + arêtes de co-occurrence)",
        "Corpus indexé": "4945 nœuds / 5593 relations (après post-traitement du 22/07/2026)",
        "Paramètres retrieval": f"mode=hybrid, top_k={graph_v3.TOP_K_LIGHTRAG}, chunk_top_k={graph_v3.CHUNK_TOP_K}, enable_rerank=False",
        "Générateur": GENERATOR_DESC,
        "Boucle agentique": "Non (1 seul appel de génération)",
    },
    {
        "Système": "Agentic GraphRAG",
        "Backend retrieval": f"LightRAG — {graph_v3.INDEX_DIR} (identique au système 2)",
        "Corpus indexé": "4945 nœuds / 5593 relations (identique au système 2)",
        "Paramètres retrieval": f"mode=hybrid, top_k={graph_v3.TOP_K_LIGHTRAG}, chunk_top_k={graph_v3.CHUNK_TOP_K}, enable_rerank=False",
        "Générateur": GENERATOR_DESC,
        "Boucle agentique": f"Oui — juge={judge_desc}, seuil_critique={graph_v3.CRITIQUE_SEUIL}, max_iterations={graph_v3.MAX_ITERATIONS}",
    },
])

display(params_table)
params_table.to_csv("Eval_agentic/ablation_study_parametres.csv", index=False)
print("✓ Eval_agentic/ablation_study_parametres.csv")

## 9. Limites et discussion

In [ ]:
best_system = comparison_table[METRICS].mean(axis=1).idxmax()
delta_vs_baseline = (comparison_table.loc["Agentic GraphRAG", METRICS] - comparison_table.loc["RAG baseline (ChromaDB)", METRICS]).round(3)
delta_vs_lightrag = (comparison_table.loc["Agentic GraphRAG", METRICS] - comparison_table.loc["LightRAG hybride (sans agent)", METRICS]).round(3)

print("=== Lecture automatique des résultats obtenus (calculée, pas supposée) ===\n")
print(f"Générateur utilisé pour les 3 systèmes : {GENERATOR_DESC}\n")
print(f"Système avec la moyenne RAGAS la plus élevée : {best_system}\n")
print("Delta Agentic GraphRAG - RAG baseline :")
print(delta_vs_baseline.to_string())
print("\nDelta Agentic GraphRAG - LightRAG hybride (sans agent) :")
print(delta_vs_lightrag.to_string())
print(f"\nTaille moyenne du contexte (système 2/3, LightRAG) : {df_details_lightrag['context_chars'].mean():.0f} caractères (max={df_details_lightrag['context_chars'].max()})")
print(f"Taille moyenne du contexte (système 1, ChromaDB)   : {df_details_baseline['context_chars'].mean():.0f} caractères (max={df_details_baseline['context_chars'].max()})")
if not (graph_v3.USE_NVIDIA and graph_v3.NVIDIA_API_KEY) and not (graph_v3.USE_GROQ and graph_v3.GROQ_API_KEY):
    print(f"Fenêtre de contexte du générateur local (num_ctx) : {graph_v3.OLLAMA_NUM_CTX} tokens — "
          f"à comparer à la taille de contexte mesurée ci-dessus")
print("""
Interprétation à adapter selon le résultat réel ci-dessus :
- Si les deltas sont POSITIFS sur les 4 métriques : la boucle agentique (critique
  + self-correct) apporte un gain mesurable au-delà du graphe seul -> résultat
  à mettre en avant tel quel dans le mémoire.
- Si LightRAG seul égale ou dépasse l'Agentic sur certaines métriques : cela
  peut s'expliquer par la taille limitée du corpus (500 documents) qui laisse
  peu de marge au SELF_CORRECT pour retrouver un meilleur contexte -> à
  documenter comme limite du corpus, pas de l'architecture.
- Si les 3 systèmes sont proches : le goulot d'étranglement est probablement
  le retrieval / la qualité du graphe plutôt que la couche agentique -> cf.
  discussion ci-dessous.
""")

### Limites identifiées

**Indexation et qualité du graphe**
- Le graphe LightRAG a été post-traité (fusion de 103 clusters de doublons, ajout de 498 arêtes de co-occurrence) mais reste **peu dense** : densité 0.00046, degré moyen 2.26, composante géante à 55.3 % des nœuds seulement (contre 43.3 % avant post-traitement). Une partie des questions peut donc rester sans chemin de graphe direct entre les entités concernées, même après amélioration.
- Le corpus indexé (500 résumés arXiv en cs.AI) est **volontairement restreint** bien adapté à un PFE mais trop petit pour que l'avantage du multi-hop agentique se manifeste pleinement (peu de chaînes de raisonnement alternatives disponibles en cas de premier échec).

**Benchmark et évaluation**
- Échantillon de 20 questions tirées des 76 questions vrai multi-hop validées par contenu (sur 340 candidates initiales)  suffisant pour dégager une tendance.


**Modèle et infrastructure**
- Le générateur effectivement utilisé (affiché en section 2 et à la cellule précédente) a sa propre limite de fenêtre de contexte ; si le contexte fusionné mesuré ci-dessus s'en approche ou la dépasse, cela peut dégrader Faithfulness indépendamment de la qualité du retrieval lui-même.
- Le juge RAGAS (métriques Faithfulness/Context Precision/Context Recall) utilise le même modèle pour les trois systèmes  cohérent pour la comparaison relative entre systèmes, mais signifie que les scores absolus dépendent aussi des capacités de jugement propres à ce modèle.

**Ce que cette étude démontre malgré ces limites**
Le protocole isole correctement la variable étudiée : système 2 et système 3 partagent strictement le même retrieval (même index, mêmes paramètres, mêmes appels) et le même générateur , toute différence entre les deux résultats est donc attribuable à la boucle agentique (critique + self-correction), et non à un effet de retrieval ou de modèle confondu.